## Make Detection with the Trained Model

In [28]:
import mediapipe as mp
import cv2
import numpy as np
import pandas as pd

import pickle

import warnings
warnings.filterwarnings('ignore')

# Drawing helpers
mp_drawing = mp.solutions.drawing_utils
mp_pose = mp.solutions.pose

### Reconstruct the input structure

In [29]:
# Determine important landmarks for staff
IMPORTANT_LMS = [
    "NOSE",
    "LEFT_SHOULDER",
    "RIGHT_SHOULDER",
    "LEFT_ELBOW",
    "RIGHT_ELBOW",
    "LEFT_WRIST",
    "RIGHT_WRIST",
    "LEFT_HIP",
    "RIGHT_HIP",
    "LEFT_KNEE",
    "RIGHT_KNEE",
    "LEFT_ANKLE",
    "RIGHT_ANKLE",
    "LEFT_HEEL",
    "RIGHT_HEEL",
    "LEFT_FOOT_INDEX",
    "RIGHT_FOOT_INDEX",
]

# Generate all columns of the data frame

HEADERS = ["label"] # Label column

for lm in IMPORTANT_LMS:
    HEADERS += [f"{lm.lower()}_x", f"{lm.lower()}_y", f"{lm.lower()}_z", f"{lm.lower()}_v"]

### Setup some important functions

In [30]:
def extract_important_keypoints(results) -> list:
    '''
    Extract important keypoints from mediapipe pose detection
    '''
    landmarks = results.pose_landmarks.landmark

    data = []
    for lm in IMPORTANT_LMS:
        keypoint = landmarks[mp_pose.PoseLandmark[lm].value]
        data.append([keypoint.x, keypoint.y, keypoint.z, keypoint.visibility])
    
    return np.array(data).flatten().tolist()


def rescale_frame(frame, percent=50):
    '''
    Rescale a frame to a certain percentage compare to its original frame
    '''
    width = int(frame.shape[1] * percent/ 100)
    height = int(frame.shape[0] * percent/ 100)
    dim = (width, height)
    return cv2.resize(frame, dim, interpolation =cv2.INTER_AREA)


# def rescale_frame_to_max(frame, max_dim=800):
#     """
#     Resize so the largest side is max_dim (keeps aspect ratio).
#     Returns original if already smaller.
#     """
#     h, w = frame.shape[:2]
#     if max(h, w) <= max_dim:
#         return frame
#     scale = max_dim / max(h, w)
#     dim = (int(w * scale), int(h * scale))
#     return cv2.resize(frame, dim, interpolation=cv2.INTER_AREA)


In [31]:
# VIDEO_TEST = "../../demo/plank_demo.mp4"
# VIDEO_TEST = "Phalakasana-2.mp4"
# VIDEO_TEST = "Phalakasana-9.mp4"
VIDEO_TEST = "sample6.mp4"
# VIDEO_TEST = "plank test.mp4"

## 1. Make detection with Scikit learn model

In [32]:
# Load model
# with open("./model/LR_model.pkl", "rb") as f:
#     sklearn_model = pickle.load(f)



# with open("./model/xgboost_model.pkl", "rb") as f:
#     model = pickle.load(f)

# # Load input scaler
# with open("./model/input_scaler.pkl", "rb") as f2:
#     input_scaler = pickle.load(f2)


with open("./model/staff_pipeline.pkl", "rb") as f:
    staff_pipeline = pickle.load(f)

# Transform prediction into class
def get_class(prediction: float) -> str:
    return {
        0: "correct",
        1: "rounded_back",
        2: "leaning_back",
        3: "bent_knees",
        4: "feet_not_flexed",
        5: "arms_not_pressing",
    }.get(prediction)


In [33]:
import sys
import os

# Add the path to the parent folder of 'angle_calculation'
sys.path.append(os.path.abspath(os.path.join(os.path.dirname(''), '../angle_calculation')))


In [34]:
from staff_angle_calculation import (
    compute_staff_shoulder_angle,
    compute_staff_elbow_angle,
    compute_staff_hip_angle,
    compute_staff_knee_angle,
)

def compute_all_angles(row):
    """
    Compute all required angles for staff pose.
    Returns a list of angles in the same order used during training.
    """
    angles = [
        compute_staff_shoulder_angle(row, side="left"),
        compute_staff_elbow_angle(row, side="left"),
        compute_staff_hip_angle(row, side="left"),
        compute_staff_knee_angle(row, side="left"),
        compute_staff_shoulder_angle(row, side="right"),
        compute_staff_elbow_angle(row, side="right"),
        compute_staff_hip_angle(row, side="right"),
        compute_staff_knee_angle(row, side="right"),
    ]
    
    return angles



In [35]:
# Angle column names (same order as compute_all_angles)
ANGLE_COLUMNS = [
    "left_shoulder_angle",
    "left_elbow_angle",
    "left_hip_angle",
    "left_knee_angle",
    "right_shoulder_angle",
    "right_elbow_angle",
    "right_hip_angle",
    "right_knee_angle",
]


In [36]:
for i, label in enumerate(staff_pipeline.named_steps['model'].classes_):
    print(f"{i} → {label}")

0 → 0
1 → 1
2 → 2
3 → 3
4 → 4
5 → 5


In [39]:
cap = cv2.VideoCapture(VIDEO_TEST)
current_stage = ""
prediction_probability_threshold = 0.6

with mp_pose.Pose(min_detection_confidence=0.5, min_tracking_confidence=0.5) as pose:
    while cap.isOpened():
        ret, image = cap.read()

        if not ret:
            break

        # Reduce size of a frame
        image = rescale_frame(image, 50)
        # image = cv2.flip(image, 1)

        # Recolor image from BGR to RGB for mediapipe
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        image.flags.writeable = False

        results = pose.process(image)

        if not results.pose_landmarks:
            print("No human found")
            continue

        # Recolor image from BGR to RGB for mediapipe
        image.flags.writeable = True
        image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)

        # Draw landmarks and connections
        mp_drawing.draw_landmarks(image, results.pose_landmarks, mp_pose.POSE_CONNECTIONS, mp_drawing.DrawingSpec(color=(244, 117, 66), thickness=2, circle_radius=2), mp_drawing.DrawingSpec(color=(245, 66, 230), thickness=2, circle_radius=1))

        # Make detection
        try:
            # Extract keypoints from frame for the input
            row = extract_important_keypoints(results)  # list of landmarks
            X_landmarks = pd.DataFrame([row], columns=HEADERS[1:])  # only landmarks columns

            # Compute all angles for this frame
            angles = compute_all_angles(X_landmarks.iloc[0])  # returns list of angles in the same order as training
            X_angles = pd.DataFrame([angles], columns=ANGLE_COLUMNS)  # ANGLE_COLUMNS = list of your angle column names

            # Combine landmarks + angles
            X_full = pd.concat([X_landmarks, X_angles], axis=1)

            # Predict with the pipeline (no need to scale manually)
            predicted_class = staff_pipeline.predict(X_full)[0]
            predicted_class = get_class(predicted_class)

            prediction_probability = staff_pipeline.predict_proba(X_full)[0]
            print(predicted_class, prediction_probability)

            # # Evaluate model prediction
            # if predicted_class == 0 and prediction_probability[prediction_probability.argmax()] >= prediction_probability_threshold:
            #     current_stage = "Correct"
            # elif predicted_class == 2 and prediction_probability[prediction_probability.argmax()] >= prediction_probability_threshold: 
            #     current_stage = "Low back"
            # elif predicted_class == 1 and prediction_probability[prediction_probability.argmax()] >= prediction_probability_threshold: 
            #     current_stage = "High back"
            # else:
            #     current_stage = "unk"
            
            if predicted_class == "correct" and prediction_probability[prediction_probability.argmax()] >= prediction_probability_threshold:
                current_stage = "Correct"
            elif predicted_class == "rounded_back" and prediction_probability[prediction_probability.argmax()] >= prediction_probability_threshold:
                current_stage = "Rounded Back"
            elif predicted_class == "leaning_back" and prediction_probability[prediction_probability.argmax()] >= prediction_probability_threshold:
                current_stage = "Leaning Back"
            elif predicted_class == "bent_knees" and prediction_probability[prediction_probability.argmax()] >= prediction_probability_threshold:
                current_stage = "Bent Knees"
            elif predicted_class == "feet_not_flexed" and prediction_probability[prediction_probability.argmax()] >= prediction_probability_threshold:
                current_stage = "Feet Not Flexed"
            elif predicted_class == "arms_not_pressing" and prediction_probability[prediction_probability.argmax()] >= prediction_probability_threshold:
                current_stage = "Arms Not Pressing"
            else:
                current_stage = "unk"
            
            # Visualization
            # Status box
            cv2.rectangle(image, (0, 0), (250, 60), (245, 117, 16), -1)

            # Display class
            cv2.putText(image, "CLASS", (95, 12), cv2.FONT_HERSHEY_COMPLEX, 0.5, (0, 0, 0), 1, cv2.LINE_AA)
            cv2.putText(image, current_stage, (90, 40), cv2.FONT_HERSHEY_COMPLEX, 1, (255, 255, 255), 2, cv2.LINE_AA)

            # Display probability
            cv2.putText(image, "PROB", (15, 12), cv2.FONT_HERSHEY_COMPLEX, 0.5, (0, 0, 0), 1, cv2.LINE_AA)
            cv2.putText(image, str(round(prediction_probability[np.argmax(prediction_probability)], 2)), (10, 40), cv2.FONT_HERSHEY_COMPLEX, 1, (255, 255, 255), 2, cv2.LINE_AA)

        except Exception as e:
            print(f"Error: {e}")
        
        cv2.imshow("CV2", image)
        
        # Press Q to close cv2 window
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    cap.release()
    cv2.destroyAllWindows()

    

    for i in range (1, 5):
        cv2.waitKey(1)
  

leaning_back [1.0757249e-02 4.0600047e-02 7.0417231e-01 2.1818573e-02 7.0246184e-05
 2.2258158e-01]
leaning_back [2.0950127e-02 2.5511583e-02 6.5626466e-01 1.8640032e-02 6.7569687e-05
 2.7856603e-01]
leaning_back [2.3574287e-02 3.3669651e-02 7.4062556e-01 2.1036154e-02 6.6396140e-05
 1.8102790e-01]
leaning_back [2.6010098e-02 3.7626863e-02 7.1444452e-01 2.2963680e-02 6.6989742e-05
 1.9888785e-01]
leaning_back [1.9706735e-02 3.1699050e-02 6.8111759e-01 1.9345935e-02 5.6436045e-05
 2.4807425e-01]
leaning_back [2.8629929e-02 3.2367330e-02 8.4853184e-01 2.4101041e-02 7.6069802e-05
 6.6293783e-02]
leaning_back [2.2987941e-02 2.9269416e-02 7.6731789e-01 2.1794302e-02 6.3578438e-05
 1.5856688e-01]
leaning_back [1.7203733e-02 3.0099833e-02 6.6210085e-01 1.8805800e-02 5.3153217e-05
 2.7173665e-01]
leaning_back [1.8963946e-02 2.8494887e-02 6.2679714e-01 1.7803060e-02 5.1935163e-05
 3.0788898e-01]
leaning_back [2.3774985e-02 3.4921154e-02 7.6815462e-01 2.1818068e-02 6.3647763e-05
 1.5126756e-01]


In [ ]:
# cap = cv2.VideoCapture(VIDEO_TEST)
# current_stage = ""
# prediction_probability_threshold = 0.82

# with mp_pose.Pose(min_detection_confidence=0.5, min_tracking_confidence=0.5) as pose:
#     while cap.isOpened():
#         ret, image = cap.read()

#         if not ret:
#             break

#         # Reduce size of a frame
#         image = rescale_frame(image, 50)
#         # image = cv2.flip(image, 1)

#         # Recolor image from BGR to RGB for mediapipe
#         image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
#         image.flags.writeable = False

#         results = pose.process(image)

#         if not results.pose_landmarks:
#             print("No human found")
#             continue

#         # Recolor image from BGR to RGB for mediapipe
#         image.flags.writeable = True
#         image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)

#         # Draw landmarks and connections
#         mp_drawing.draw_landmarks(image, results.pose_landmarks, mp_pose.POSE_CONNECTIONS, mp_drawing.DrawingSpec(color=(244, 117, 66), thickness=2, circle_radius=2), mp_drawing.DrawingSpec(color=(245, 66, 230), thickness=2, circle_radius=1))

#         # Make detection
#         try:
#             # Extract keypoints from frame for the input
#             row = extract_important_keypoints(results)
#             X = pd.DataFrame([row], columns=HEADERS[1:])
#             # Scale features (keep as DataFrame)
#             X_scaled = pd.DataFrame(input_scaler.transform(X), columns=X.columns)

#             # Make prediction and its probability
#             predicted_class = sklearn_model.predict(X_scaled)[0]
#             predicted_class = get_class(predicted_class)
#             prediction_probability = sklearn_model.predict_proba(X_scaled)[0]
#             # print(predicted_class, prediction_probability)

#             # Evaluate model prediction
#             if predicted_class == "C" and prediction_probability[prediction_probability.argmax()] >= prediction_probability_threshold:
#                 current_stage = "Correct"
#             elif predicted_class == "L" and prediction_probability[prediction_probability.argmax()] >= prediction_probability_threshold: 
#                 current_stage = "Low back"
#             elif predicted_class == "H" and prediction_probability[prediction_probability.argmax()] >= prediction_probability_threshold: 
#                 current_stage = "High back"
#             else:
#                 current_stage = "unk"
            
#             # Visualization
#             # Status box
#             cv2.rectangle(image, (0, 0), (250, 60), (245, 117, 16), -1)

#             # Display class
#             cv2.putText(image, "CLASS", (95, 12), cv2.FONT_HERSHEY_COMPLEX, 0.5, (0, 0, 0), 1, cv2.LINE_AA)
#             cv2.putText(image, current_stage, (90, 40), cv2.FONT_HERSHEY_COMPLEX, 1, (255, 255, 255), 2, cv2.LINE_AA)

#             # Display probability
#             cv2.putText(image, "PROB", (15, 12), cv2.FONT_HERSHEY_COMPLEX, 0.5, (0, 0, 0), 1, cv2.LINE_AA)
#             cv2.putText(image, str(round(prediction_probability[np.argmax(prediction_probability)], 2)), (10, 40), cv2.FONT_HERSHEY_COMPLEX, 1, (255, 255, 255), 2, cv2.LINE_AA)

#         except Exception as e:
#             print(f"Error: {e}")
        
#         cv2.imshow("CV2", image)
        
#         # Press Q to close cv2 window
#         if cv2.waitKey(1) & 0xFF == ord('q'):
#             break

#     cap.release()
#     cv2.destroyAllWindows()

    

#     for i in range (1, 5):
#         cv2.waitKey(1)
  

In [ ]:
# cap = cv2.VideoCapture(VIDEO_TEST)
# current_stage = ""
# prediction_probability_threshold = 0.82

# with mp_pose.Pose(min_detection_confidence=0.5, min_tracking_confidence=0.5) as pose:
#     while cap.isOpened():
#         ret, image = cap.read()

#         if not ret:
#             break

#         # Reduce size of a frame
#         image = rescale_frame(image, 50)
#         # image = cv2.flip(image, 1)

#         # Recolor image from BGR to RGB for mediapipe
#         image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
#         image.flags.writeable = False

#         results = pose.process(image)

#         if not results.pose_landmarks:
#             print("No human found")
#             continue

#         # Recolor image from BGR to RGB for mediapipe
#         image.flags.writeable = True
#         image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)

#         # Draw landmarks and connections
#         mp_drawing.draw_landmarks(image, results.pose_landmarks, mp_pose.POSE_CONNECTIONS, mp_drawing.DrawingSpec(color=(244, 117, 66), thickness=2, circle_radius=2), mp_drawing.DrawingSpec(color=(245, 66, 230), thickness=2, circle_radius=1))

#         # Make detection
#         try:
#             # Extract keypoints from frame for the input
#             row = extract_important_keypoints(results)
#             X = pd.DataFrame([row], columns=HEADERS[1:])
#             X = pd.DataFrame(input_scaler.transform(X))

#             # Make prediction and its probability
#             predicted_class = sklearn_model.predict(X)[0]
#             predicted_class = get_class(predicted_class)
#             prediction_probability = sklearn_model.predict_proba(X)[0]
#             # print(predicted_class, prediction_probability)

#             # Evaluate model prediction
#             if predicted_class == "C" and prediction_probability[prediction_probability.argmax()] >= prediction_probability_threshold:
#                 current_stage = "Correct"
#             elif predicted_class == "L" and prediction_probability[prediction_probability.argmax()] >= prediction_probability_threshold: 
#                 current_stage = "Low back"
#             elif predicted_class == "H" and prediction_probability[prediction_probability.argmax()] >= prediction_probability_threshold: 
#                 current_stage = "High back"
#             else:
#                 current_stage = "unk"
            
#             # Visualization
#             # Status box
#             cv2.rectangle(image, (0, 0), (250, 60), (245, 117, 16), -1)

#             # Display class
#             cv2.putText(image, "CLASS", (95, 12), cv2.FONT_HERSHEY_COMPLEX, 0.5, (0, 0, 0), 1, cv2.LINE_AA)
#             cv2.putText(image, current_stage, (90, 40), cv2.FONT_HERSHEY_COMPLEX, 1, (255, 255, 255), 2, cv2.LINE_AA)

#             # Display probability
#             cv2.putText(image, "PROB", (15, 12), cv2.FONT_HERSHEY_COMPLEX, 0.5, (0, 0, 0), 1, cv2.LINE_AA)
#             cv2.putText(image, str(round(prediction_probability[np.argmax(prediction_probability)], 2)), (10, 40), cv2.FONT_HERSHEY_COMPLEX, 1, (255, 255, 255), 2, cv2.LINE_AA)

#         except Exception as e:
#             print(f"Error: {e}")
        
#         cv2.imshow("CV2", image)
        
#         # Press Q to close cv2 window
#         if cv2.waitKey(1) & 0xFF == ord('q'):
#             break

#     cap.release()
#     cv2.destroyAllWindows()

    

#     for i in range (1, 5):
#         cv2.waitKey(1)
  

## 2. Make detection with Deep Learning Model

In [ ]:
# Load model
with open("./model/plank_dp.pkl", "rb") as f:
    deep_learning_model = pickle.load(f)

In [ ]:
cap = cv2.VideoCapture(VIDEO_TEST)
current_stage = ""
prediction_probability_threshold = 0.8

with mp_pose.Pose(min_detection_confidence=0.5, min_tracking_confidence=0.5) as pose:
    while cap.isOpened():
        ret, image = cap.read()

        if not ret:
            break

        # Reduce size of a frame
        image = rescale_frame(image, 50)

        # Recolor image from BGR to RGB for mediapipe
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        image.flags.writeable = False

        results = pose.process(image)

        if not results.pose_landmarks:
            print("No human found")
            continue

        # Recolor image from BGR to RGB for mediapipe
        image.flags.writeable = True
        image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)

        # Draw landmarks and connections
        mp_drawing.draw_landmarks(image, results.pose_landmarks, mp_pose.POSE_CONNECTIONS, mp_drawing.DrawingSpec(color=(244, 117, 66), thickness=2, circle_radius=2), mp_drawing.DrawingSpec(color=(245, 66, 230), thickness=2, circle_radius=1))

        # Make detection
        try:
            # Extract keypoints from frame for the input
            row = extract_important_keypoints(results)
            X = pd.DataFrame([row, ], columns=HEADERS[1:])
            X = pd.DataFrame(input_scaler.transform(X))
            

            # Make prediction and its probability
            prediction = deep_learning_model.predict(X)
            predicted_class = np.argmax(prediction, axis=1)[0]

            prediction_probability = max(prediction.tolist()[0])
            # print(X)

            # Evaluate model prediction
            if predicted_class == 0 and prediction_probability >= prediction_probability_threshold:
                current_stage = "Correct"
            elif predicted_class == 2 and prediction_probability >= prediction_probability_threshold: 
                current_stage = "Low back"
            elif predicted_class == 1 and prediction_probability >= prediction_probability_threshold: 
                current_stage = "High back"
            else:
                current_stage = "Unknown"

            # Visualization
            # Status box
            cv2.rectangle(image, (0, 0), (550, 60), (245, 117, 16), -1)

            # # Display class
            cv2.putText(image, "DETECTION", (95, 12), cv2.FONT_HERSHEY_COMPLEX, 0.5, (0, 0, 0), 1, cv2.LINE_AA)
            cv2.putText(image, current_stage, (90, 40), cv2.FONT_HERSHEY_COMPLEX, 1, (255, 255, 255), 2, cv2.LINE_AA)

            # # Display class
            cv2.putText(image, "CLASS", (350, 12), cv2.FONT_HERSHEY_COMPLEX, 0.5, (0, 0, 0), 1, cv2.LINE_AA)
            cv2.putText(image, str(predicted_class), (345, 40), cv2.FONT_HERSHEY_COMPLEX, 1, (255, 255, 255), 2, cv2.LINE_AA)

            # # Display probability
            cv2.putText(image, "PROB", (15, 12), cv2.FONT_HERSHEY_COMPLEX, 0.5, (0, 0, 0), 1, cv2.LINE_AA)
            cv2.putText(image, str(round(prediction_probability, 2)), (10, 40), cv2.FONT_HERSHEY_COMPLEX, 1, (255, 255, 255), 2, cv2.LINE_AA)

        except Exception as e:
            print(f"Error: {e}")
        
        cv2.imshow("CV2", image)
        
        # Press Q to close cv2 window
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    cap.release()
    cv2.destroyAllWindows()

    for i in range (1, 5):
        cv2.waitKey(1)
  

In [ ]:
X